# Instance-Level BERTimbau Training

This notebook trains the instance-level BERTimbau Large baseline on the six validated pair-controlled Puntuguese splits using multiple model seeds. Each text is treated independently during training with cross-entropy loss. Evaluation includes conventional instance-level metrics and true-pair metrics reconstructed from the test IDs. Results are stored separately for every split seed and model seed, followed by aggregated summaries across runs.

In [1]:
from pathlib import Path
import gc
import inspect
import json
import platform
import random
import shutil
import sys
import time

import numpy as np
import pandas as pd
import torch
import transformers

from IPython.display import display
from sklearn import __version__ as sklearn_version
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from transformers.utils import logging as hf_logging

/home/avelar/pair-aware/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"

SPLIT_SEEDS = [13, 21, 40, 42, 73, 101]
MODEL_SEEDS = [40, 123, 456]
SPLIT_NAMES = ["train", "validation", "test"]

EXPECTED_SPLIT_COUNTS = {
    "train": 3990,
    "validation": 570,
    "test": 1140,
}

EXPECTED_PAIR_COUNTS = {
    "train": 1995,
    "validation": 285,
    "test": 570,
}

EXPECTED_CLASS_COUNTS = {
    "train": {0: 1995, 1: 1995},
    "validation": {0: 285, 1: 285},
    "test": {0: 570, 1: 570},
}

MAX_LENGTH = 256
NUM_EPOCHS = 6
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2

KEEP_CHECKPOINTS = False
SKIP_COMPLETED_RUNS = True

ID2LABEL = {
    0: "0",
    1: "1",
}

LABEL2ID = {
    "0": 0,
    "1": 1,
}

In [ ]:
def find_project_root(start_path=None):
    current = Path(start_path or Path.cwd()).resolve()

    while True:
        if (current / "data" / "pair_controlled").is_dir():
            return current

        if current == current.parent:
            break

        current = current.parent

    raise FileNotFoundError(
        "Could not locate the project root containing data/pair_controlled."
    )

In [ ]:
PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "pair_controlled"
RESULTS_DIR = PROJECT_ROOT / "results" / "instance_level"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", PROJECT_ROOT)
print("Pair-controlled data:", DATA_DIR)
print("Results:", RESULTS_DIR)

In [ ]:
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn_version)
print("CUDA available:", torch.cuda.is_available())
print("CUDA:", torch.version.cuda)
print("Split seeds:", SPLIT_SEEDS)
print("Model seeds:", MODEL_SEEDS)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("cuDNN:", torch.backends.cudnn.version())

In [ ]:
def validate_transformers_api():
    parameters = set(
        inspect.signature(
            TrainingArguments.__init__
        ).parameters
    )

    required_parameters = {
        "output_dir",
        "num_train_epochs",
        "learning_rate",
        "lr_scheduler_type",
        "per_device_train_batch_size",
        "per_device_eval_batch_size",
        "weight_decay",
        "warmup_steps",
        "max_grad_norm",
        "logging_strategy",
        "eval_strategy",
        "save_strategy",
        "load_best_model_at_end",
        "metric_for_best_model",
        "greater_is_better",
        "save_total_limit",
        "report_to",
        "fp16",
        "seed",
        "data_seed",
    }

    missing_parameters = required_parameters - parameters

    if missing_parameters:
        raise RuntimeError(
            "The installed Transformers version does not expose the "
            "TrainingArguments API expected by this notebook. "
            f"Missing parameters: {sorted(missing_parameters)}"
        )

    print(
        "TrainingArguments API validated for Transformers",
        transformers.__version__,
    )

In [ ]:
validate_transformers_api()

In [ ]:
def set_model_seed(model_seed):
    random.seed(model_seed)
    np.random.seed(model_seed)
    torch.manual_seed(model_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(model_seed)
        torch.cuda.manual_seed_all(model_seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def read_jsonl(file_path):
    rows = []

    with Path(file_path).open(
        "r",
        encoding="utf-8",
    ) as file:
        for line in file:
            line = line.strip()

            if line:
                rows.append(
                    json.loads(line)
                )

    return pd.DataFrame(rows)

In [ ]:
def parse_pair_id(example_id):
    parts = str(example_id).rsplit(".", 1)

    if len(parts) != 2:
        raise ValueError(
            f"Invalid example ID: {example_id}"
        )

    pair_id, suffix = parts

    if not pair_id or suffix not in {"H", "N"}:
        raise ValueError(
            f"Invalid example ID: {example_id}"
        )

    return pair_id, suffix

In [ ]:
def load_split_directory(split_dir):
    split_dir = Path(split_dir)

    paths = {
        split_name: split_dir / f"{split_name}.jsonl"
        for split_name in SPLIT_NAMES
    }

    for split_name, path in paths.items():
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing {split_name} file: {path}"
            )

    return {
        split_name: read_jsonl(path)
        for split_name, path in paths.items()
    }

In [ ]:
def validate_input_splits(split_data, run_name):
    pair_sets = {}
    all_ids = []

    for split_name in SPLIT_NAMES:
        split_df = split_data[split_name].copy()

        required_columns = {
            "id",
            "text",
            "label",
        }

        missing_columns = required_columns - set(
            split_df.columns
        )

        if missing_columns:
            raise ValueError(
                f"{run_name}/{split_name}: missing columns "
                f"{sorted(missing_columns)}."
            )

        if len(split_df) != EXPECTED_SPLIT_COUNTS[split_name]:
            raise ValueError(
                f"{run_name}/{split_name}: expected "
                f"{EXPECTED_SPLIT_COUNTS[split_name]} examples, "
                f"found {len(split_df)}."
            )

        split_df["label"] = split_df["label"].astype(int)

        observed_classes = (
            split_df["label"]
            .value_counts()
            .sort_index()
            .to_dict()
        )

        if observed_classes != EXPECTED_CLASS_COUNTS[split_name]:
            raise ValueError(
                f"{run_name}/{split_name}: expected class distribution "
                f"{EXPECTED_CLASS_COUNTS[split_name]}, "
                f"found {observed_classes}."
            )

        if split_df["id"].duplicated().any():
            raise ValueError(
                f"{run_name}/{split_name}: duplicated IDs."
            )

        pair_members = {}

        for example_id in split_df["id"].astype(str):
            pair_id, suffix = parse_pair_id(example_id)
            pair_members.setdefault(pair_id, []).append(suffix)

        invalid_pairs = {
            pair_id: suffixes
            for pair_id, suffixes in pair_members.items()
            if sorted(suffixes) != ["H", "N"]
        }

        if invalid_pairs:
            raise ValueError(
                f"{run_name}/{split_name}: invalid H/N pairs found."
            )

        if len(pair_members) != EXPECTED_PAIR_COUNTS[split_name]:
            raise ValueError(
                f"{run_name}/{split_name}: expected "
                f"{EXPECTED_PAIR_COUNTS[split_name]} pairs, "
                f"found {len(pair_members)}."
            )

        pair_sets[split_name] = set(pair_members)
        all_ids.extend(
            split_df["id"].astype(str).tolist()
        )

    if len(all_ids) != 5700:
        raise ValueError(
            f"{run_name}: expected 5700 total IDs."
        )

    if len(set(all_ids)) != 5700:
        raise ValueError(
            f"{run_name}: IDs overlap across splits."
        )

    crossing_pairs = (
        pair_sets["train"].intersection(pair_sets["validation"])
        | pair_sets["train"].intersection(pair_sets["test"])
        | pair_sets["validation"].intersection(pair_sets["test"])
    )

    if crossing_pairs:
        raise ValueError(
            f"{run_name}: pair IDs overlap across splits."
        )

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print(
    "Tokenizer loaded:",
    MODEL_NAME,
)

In [ ]:
class PunDataset(Dataset):
    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length,
    ):
        self.texts = (
            dataframe["text"]
            .astype(str)
            .tolist()
        )

        self.labels = (
            dataframe["label"]
            .astype(int)
            .tolist()
        )

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(
                self.labels[index],
                dtype=torch.long,
            ),
        }

        if "token_type_ids" in encoding:
            item["token_type_ids"] = (
                encoding["token_type_ids"]
                .squeeze(0)
            )

        return item

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1,
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predictions,
        ),
        "precision_macro": precision_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "f1_macro": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "precision_weighted": precision_score(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        ),
        "recall_weighted": recall_score(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        ),
    }

In [ ]:
def softmax_numpy(logits):
    shifted = logits - np.max(
        logits,
        axis=1,
        keepdims=True,
    )

    exponentials = np.exp(shifted)

    return exponentials / np.sum(
        exponentials,
        axis=1,
        keepdims=True,
    )

In [ ]:
def evaluate_instance_predictions(
    y_true,
    y_pred,
):
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["non_pun", "pun"],
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "precision_non_pun": float(
            report_dict["non_pun"]["precision"]
        ),
        "recall_non_pun": float(
            report_dict["non_pun"]["recall"]
        ),
        "f1_non_pun": float(
            report_dict["non_pun"]["f1-score"]
        ),
        "precision_pun": float(
            report_dict["pun"]["precision"]
        ),
        "recall_pun": float(
            report_dict["pun"]["recall"]
        ),
        "f1_pun": float(
            report_dict["pun"]["f1-score"]
        ),
        "precision_macro": float(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "recall_macro": float(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "precision_weighted": float(
            precision_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),
        "recall_weighted": float(
            recall_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),
        "f1_weighted": float(
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }

    return metrics, report_dict, cm

In [ ]:
def build_instance_predictions(
    test_df,
    logits,
):
    probabilities = softmax_numpy(logits)
    predictions = np.argmax(
        logits,
        axis=1,
    )

    rows = []

    for index, row in test_df.reset_index(drop=True).iterrows():
        pair_id, suffix = parse_pair_id(
            row["id"]
        )

        rows.append(
            {
                "id": str(row["id"]),
                "pair_id": pair_id,
                "suffix": suffix,
                "true_label": int(row["label"]),
                "predicted_label": int(predictions[index]),
                "logit_non_pun": float(logits[index, 0]),
                "logit_pun": float(logits[index, 1]),
                "pun_score": float(
                    logits[index, 1]
                    - logits[index, 0]
                ),
                "probability_non_pun": float(
                    probabilities[index, 0]
                ),
                "probability_pun": float(
                    probabilities[index, 1]
                ),
            }
        )

    return pd.DataFrame(rows)

In [ ]:
def evaluate_true_pairs(instance_predictions):
    pair_rows = []

    grouped = instance_predictions.groupby(
        "pair_id",
        sort=True,
    )

    for pair_id, pair_df in grouped:
        if len(pair_df) != 2:
            raise ValueError(
                f"Pair {pair_id} does not contain exactly two instances."
            )

        suffixes = set(
            pair_df["suffix"].tolist()
        )

        if suffixes != {"H", "N"}:
            raise ValueError(
                f"Pair {pair_id} does not contain one H and one N instance."
            )

        pun_row = pair_df.loc[
            pair_df["suffix"] == "H"
        ].iloc[0]

        non_pun_row = pair_df.loc[
            pair_df["suffix"] == "N"
        ].iloc[0]

        margin = float(
            pun_row["pun_score"]
            - non_pun_row["pun_score"]
        )

        ranking_correct = bool(
            margin > 0
        )

        ranking_tie = bool(
            margin == 0
        )

        exact_match = bool(
            pun_row["predicted_label"] == 1
            and non_pun_row["predicted_label"] == 0
        )

        pair_rows.append(
            {
                "pair_id": pair_id,
                "pun_id": pun_row["id"],
                "non_pun_id": non_pun_row["id"],
                "pun_predicted_label": int(
                    pun_row["predicted_label"]
                ),
                "non_pun_predicted_label": int(
                    non_pun_row["predicted_label"]
                ),
                "pun_score": float(
                    pun_row["pun_score"]
                ),
                "non_pun_score": float(
                    non_pun_row["pun_score"]
                ),
                "pun_probability": float(
                    pun_row["probability_pun"]
                ),
                "non_pun_probability": float(
                    non_pun_row["probability_pun"]
                ),
                "pair_margin": margin,
                "ranking_correct": ranking_correct,
                "ranking_tie": ranking_tie,
                "exact_match": exact_match,
            }
        )

    pair_predictions = pd.DataFrame(
        pair_rows
    )

    metrics = {
        "pair_count": int(
            len(pair_predictions)
        ),
        "pair_accuracy": float(
            pair_predictions["ranking_correct"].mean()
        ),
        "pair_exact_match": float(
            pair_predictions["exact_match"].mean()
        ),
        "pair_ties": int(
            pair_predictions["ranking_tie"].sum()
        ),
        "mean_pair_margin": float(
            pair_predictions["pair_margin"].mean()
        ),
        "median_pair_margin": float(
            pair_predictions["pair_margin"].median()
        ),
        "std_pair_margin": float(
            pair_predictions["pair_margin"].std(ddof=1)
        ),
        "min_pair_margin": float(
            pair_predictions["pair_margin"].min()
        ),
        "max_pair_margin": float(
            pair_predictions["pair_margin"].max()
        ),
    }

    return metrics, pair_predictions

In [ ]:
def load_fresh_model(model_seed):
    set_model_seed(
        model_seed
    )

    previous_verbosity = (
        hf_logging.get_verbosity()
    )

    hf_logging.set_verbosity_error()

    try:
        model = (
            AutoModelForSequenceClassification.from_pretrained(
                MODEL_NAME,
                num_labels=2,
                id2label=ID2LABEL,
                label2id=LABEL2ID,
            )
        )
    finally:
        hf_logging.set_verbosity(
            previous_verbosity
        )

    return model

In [ ]:
def build_training_arguments(
    checkpoint_dir,
    model_seed,
):
    return TrainingArguments(
        output_dir=str(
            checkpoint_dir
        ),
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="linear",
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_RATIO,
        max_grad_norm=MAX_GRAD_NORM,
        logging_strategy="epoch",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=model_seed,
        data_seed=model_seed,
    )

In [ ]:
def prepare_run_directory(output_dir):
    output_dir = Path(
        output_dir
    )

    checkpoint_dir = (
        output_dir
        / "checkpoints"
    )

    if checkpoint_dir.exists():
        shutil.rmtree(
            checkpoint_dir
        )

    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    return checkpoint_dir

In [ ]:
def build_run_metadata(
    split_seed,
    model_seed,
    split_dir,
    checkpoint_dir,
    trainer,
    train_df,
    validation_df,
    test_df,
):
    return {
        "method": "instance_level",
        "model": MODEL_NAME,
        "split_seed": split_seed,
        "model_seed": model_seed,
        "input_directory": str(
            Path(split_dir).resolve()
        ),
        "checkpoint_directory": str(
            Path(checkpoint_dir).resolve()
        ),
        "best_model_checkpoint": (
            trainer.state.best_model_checkpoint
        ),
        "checkpoints_retained": KEEP_CHECKPOINTS,
        "train_examples": int(
            len(train_df)
        ),
        "validation_examples": int(
            len(validation_df)
        ),
        "test_examples": int(
            len(test_df)
        ),
        "max_length": MAX_LENGTH,
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "lr_scheduler_type": "linear",
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "max_grad_norm": MAX_GRAD_NORM,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "metric_for_best_model": "f1_macro",
        "fp16": bool(
            torch.cuda.is_available()
        ),
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn_version,
        "cuda_available": bool(
            torch.cuda.is_available()
        ),
        "cuda_version": torch.version.cuda,
        "gpu": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
        "cudnn": (
            torch.backends.cudnn.version()
            if torch.cuda.is_available()
            else None
        ),
    }

In [ ]:
def save_run_outputs(
    output_dir,
    metrics,
    report_dict,
    cm,
    metadata,
    training_history,
    instance_predictions,
    pair_predictions,
):
    output_dir = Path(
        output_dir
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    pd.DataFrame(
        cm,
        index=[
            "true_non_pun",
            "true_pun",
        ],
        columns=[
            "pred_non_pun",
            "pred_pun",
        ],
    ).to_csv(
        output_dir / "confusion_matrix.csv",
        encoding="utf-8",
    )

    pd.DataFrame(
        report_dict
    ).T.to_csv(
        output_dir / "classification_report.csv",
        encoding="utf-8",
    )

    pd.DataFrame(
        training_history
    ).to_csv(
        output_dir / "training_history.csv",
        index=False,
        encoding="utf-8",
    )

    instance_predictions.to_csv(
        output_dir / "instance_predictions.csv",
        index=False,
        encoding="utf-8",
    )

    pair_predictions.to_csv(
        output_dir / "pair_predictions.csv",
        index=False,
        encoding="utf-8",
    )

    with (
        output_dir / "metrics.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metrics,
            file,
            ensure_ascii=False,
            indent=2,
        )

    with (
        output_dir / "metadata.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metadata,
            file,
            ensure_ascii=False,
            indent=2,
        )

In [ ]:
def clear_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
def load_existing_run(output_dir):
    output_dir = Path(
        output_dir
    )

    metrics_path = (
        output_dir
        / "metrics.json"
    )

    metadata_path = (
        output_dir
        / "metadata.json"
    )

    if not (
        metrics_path.is_file()
        and metadata_path.is_file()
    ):
        return None

    with metrics_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        metrics = json.load(file)

    with metadata_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        metadata = json.load(file)

    return {
        "method": metadata["method"],
        "split_seed": metadata["split_seed"],
        "model_seed": metadata["model_seed"],
        **metrics,
    }

In [ ]:
def run_instance_level(
    split_seed,
    model_seed,
):
    split_dir = (
        DATA_DIR
        / f"seed_{split_seed}"
    )

    output_dir = (
        RESULTS_DIR
        / f"split_{split_seed}"
        / f"model_seed_{model_seed}"
    )

    if SKIP_COMPLETED_RUNS:
        existing_result = load_existing_run(
            output_dir
        )

        if existing_result is not None:
            print(
                "Skipping completed run:",
                f"split={split_seed}",
                f"model_seed={model_seed}",
            )
            return existing_result

    start_time = time.time()

    set_model_seed(
        model_seed
    )

    split_data = load_split_directory(
        split_dir
    )

    validate_input_splits(
        split_data,
        f"split_{split_seed}",
    )

    train_df = split_data["train"]
    validation_df = split_data["validation"]
    test_df = split_data["test"]

    train_dataset = PunDataset(
        dataframe=train_df,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    validation_dataset = PunDataset(
        dataframe=validation_df,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    test_dataset = PunDataset(
        dataframe=test_df,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint_dir = prepare_run_directory(
        output_dir
    )

    model = load_fresh_model(
        model_seed
    )

    training_args = build_training_arguments(
        checkpoint_dir=checkpoint_dir,
        model_seed=model_seed,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE
            )
        ],
    )

    train_output = trainer.train()

    prediction_output = trainer.predict(
        test_dataset
    )

    logits = np.asarray(
        prediction_output.predictions
    )

    y_true = np.asarray(
        prediction_output.label_ids
    ).astype(int)

    y_pred = np.argmax(
        logits,
        axis=1,
    ).astype(int)

    (
        instance_metrics,
        report_dict,
        cm,
    ) = evaluate_instance_predictions(
        y_true,
        y_pred,
    )

    instance_predictions = build_instance_predictions(
        test_df=test_df,
        logits=logits,
    )

    (
        pair_metrics,
        pair_predictions,
    ) = evaluate_true_pairs(
        instance_predictions
    )

    metrics = {
        **instance_metrics,
        **pair_metrics,
    }

    prediction_metrics = prediction_output.metrics

    if "test_loss" in prediction_metrics:
        metrics["test_loss"] = float(
            prediction_metrics["test_loss"]
        )

    metrics["best_validation_f1_macro"] = (
        None
        if trainer.state.best_metric is None
        else float(
            trainer.state.best_metric
        )
    )

    metrics["train_runtime"] = float(
        train_output.metrics.get(
            "train_runtime",
            0.0,
        )
    )

    metrics["total_runtime"] = float(
        time.time() - start_time
    )

    metadata = build_run_metadata(
        split_seed=split_seed,
        model_seed=model_seed,
        split_dir=split_dir,
        checkpoint_dir=checkpoint_dir,
        trainer=trainer,
        train_df=train_df,
        validation_df=validation_df,
        test_df=test_df,
    )

    save_run_outputs(
        output_dir=output_dir,
        metrics=metrics,
        report_dict=report_dict,
        cm=cm,
        metadata=metadata,
        training_history=trainer.state.log_history,
        instance_predictions=instance_predictions,
        pair_predictions=pair_predictions,
    )

    print("=" * 80)
    print("Method: instance_level")
    print("Split seed:", split_seed)
    print("Model seed:", model_seed)

    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=["non_pun", "pun"],
            digits=4,
            zero_division=0,
        )
    )

    print(
        "Accuracy:",
        f"{metrics['accuracy']:.6f}",
    )

    print(
        "Macro-F1:",
        f"{metrics['f1_macro']:.6f}",
    )

    print(
        "Pair Accuracy:",
        f"{metrics['pair_accuracy']:.6f}",
    )

    print(
        "Pair Exact Match:",
        f"{metrics['pair_exact_match']:.6f}",
    )

    print(
        "Mean Pair Margin:",
        f"{metrics['mean_pair_margin']:.6f}",
    )

    result = {
        "method": "instance_level",
        "split_seed": split_seed,
        "model_seed": model_seed,
        **metrics,
    }

    if not KEEP_CHECKPOINTS:
        shutil.rmtree(
            checkpoint_dir,
            ignore_errors=True,
        )

    del prediction_output
    del train_dataset
    del validation_dataset
    del test_dataset
    del trainer
    del model

    clear_memory()

    return result

In [ ]:
all_results = []

for model_seed in MODEL_SEEDS:
    for split_seed in SPLIT_SEEDS:
        result = run_instance_level(
            split_seed=split_seed,
            model_seed=model_seed,
        )

        all_results.append(
            result
        )

all_results_df = pd.DataFrame(
    all_results
).sort_values(
    ["split_seed", "model_seed"]
).reset_index(
    drop=True
)

display(
    all_results_df
)

all_results_df.to_csv(
    RESULTS_DIR / "runs.csv",
    index=False,
    encoding="utf-8",
)

In [ ]:
SUMMARY_METRICS = [
    "accuracy",
    "precision_non_pun",
    "recall_non_pun",
    "f1_non_pun",
    "precision_pun",
    "recall_pun",
    "f1_pun",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "precision_weighted",
    "recall_weighted",
    "f1_weighted",
    "pair_accuracy",
    "pair_exact_match",
    "mean_pair_margin",
    "median_pair_margin",
    "std_pair_margin",
    "tp",
    "tn",
    "fp",
    "fn",
]

In [ ]:
split_summary_rows = []

for split_seed in SPLIT_SEEDS:
    split_df = all_results_df.loc[
        all_results_df["split_seed"]
        == split_seed
    ]

    row = {
        "method": "instance_level",
        "split_seed": split_seed,
        "model_seed_runs": len(
            split_df
        ),
    }

    for metric in SUMMARY_METRICS:
        row[f"{metric}_mean"] = float(
            split_df[metric].mean()
        )

        row[f"{metric}_std"] = float(
            split_df[metric].std(ddof=1)
        )

    split_summary_rows.append(
        row
    )

split_summary_df = pd.DataFrame(
    split_summary_rows
)

display(
    split_summary_df
)

split_summary_df.to_csv(
    RESULTS_DIR / "summary_by_split.csv",
    index=False,
    encoding="utf-8",
)

In [ ]:
model_seed_summary_rows = []

for model_seed in MODEL_SEEDS:
    seed_df = all_results_df.loc[
        all_results_df["model_seed"]
        == model_seed
    ]

    row = {
        "method": "instance_level",
        "model_seed": model_seed,
        "split_runs": len(
            seed_df
        ),
    }

    for metric in SUMMARY_METRICS:
        row[f"{metric}_mean"] = float(
            seed_df[metric].mean()
        )

        row[f"{metric}_std"] = float(
            seed_df[metric].std(ddof=1)
        )

    model_seed_summary_rows.append(
        row
    )

model_seed_summary_df = pd.DataFrame(
    model_seed_summary_rows
)

display(
    model_seed_summary_df
)

model_seed_summary_df.to_csv(
    RESULTS_DIR / "summary_by_model_seed.csv",
    index=False,
    encoding="utf-8",
)

In [ ]:
overall_summary = {
    "method": "instance_level",
    "split_count": len(
        SPLIT_SEEDS
    ),
    "model_seed_count": len(
        MODEL_SEEDS
    ),
    "total_runs": len(
        all_results_df
    ),
}

for metric in SUMMARY_METRICS:
    split_metric = (
        split_summary_df[
            f"{metric}_mean"
        ]
    )

    overall_summary[
        f"{metric}_mean"
    ] = float(
        split_metric.mean()
    )

    overall_summary[
        f"{metric}_std"
    ] = float(
        split_metric.std(ddof=1)
    )

overall_summary_df = pd.DataFrame(
    [overall_summary]
)

display(
    overall_summary_df
)

overall_summary_df.to_csv(
    RESULTS_DIR / "summary_overall.csv",
    index=False,
    encoding="utf-8",
)